## A

State the input, output and clinical question in your own words:

The input of this task is greyscale images, FINISH THIS

Import Libraries

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from pathlib import Path
import sys
import importlib.util
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, TensorDataset
from torchvision.transforms import v2, InterpolationMode
from torchvision import models

RANDOM_SEED_VALUE = 100
np.random.default_rng(RANDOM_SEED_VALUE)
torch.manual_seed(RANDOM_SEED_VALUE)


In [29]:
#Get the labelled CSV file as a Pandas dataframe
df = pd.read_csv("assignment1supportingfiles/assignment1_labels.csv")

mapping = pd.read_csv("assignment1supportingfiles/rsna_to_nih_mapping.csv")

print(f"Labels CSV file: Number of rows: {len(df)}")
#Drop the irrelevant columns
labels_df = df[["patientId", "Target", "StudyInstanceUID"]]


print(f"Number of duplicate patient Ids: {labels_df['patientId'].duplicated().sum()}")

labels_df = labels_df.drop_duplicates(subset="patientId")
print(f"Number of unique examinations and source patients: {len(labels_df)}")

print(f"Class distributions by count: {labels_df["Target"].value_counts()}")

print(f"Class distributions by percentage: {labels_df["Target"].value_counts(normalize=True) * 100}")


# Set train test split sizes
train_percentage = 0.8
test_percentage = validation_percentage = (1 - train_percentage) / 2



labels_df = pd.merge(labels_df,mapping, how="inner", on="patientId", validate="1:1" )
labels_df = labels_df[["patientId", "nih_patient_id", "Target", "nih_image_id"]]
print(labels_df.head())



Labels CSV file: Number of rows: 28989
Number of duplicate patient Ids: 3305
Number of unique examinations and source patients: 25684
Class distributions by count: Target
0    20025
1     5659
Name: count, dtype: int64
Class distributions by percentage: Target
0    77.966828
1    22.033172
Name: proportion, dtype: float64
                                           patientId  nih_patient_id  Target  \
0  1.2.276.0.7230010.3.1.4.8323329.5872.151787431...           20276       0   
1  1.2.276.0.7230010.3.1.4.8323329.15466.15178743...            3755       0   
2  1.2.276.0.7230010.3.1.4.8323329.10973.15178743...            6774       1   
3  1.2.276.0.7230010.3.1.4.8323329.22658.15178744...            4413       0   
4  1.2.276.0.7230010.3.1.4.8323329.18672.15178744...            8037       1   

       nih_image_id  
0  00020276_000.png  
1  00003755_000.png  
2  00006774_005.png  
3  00004413_002.png  
4  00008037_002.png  


## B

In [30]:
sgkf = StratifiedGroupKFold(n_splits=round(1 / test_percentage), random_state=RANDOM_SEED_VALUE, shuffle=True)
train_id, test_id = next(sgkf.split(labels_df, labels_df["Target"], groups=labels_df["nih_patient_id"]))

train_val_df = labels_df.iloc[train_id]
test_df = labels_df.iloc[test_id].copy()

#Do it again to get validation


sgkf = StratifiedGroupKFold(n_splits=round((1 - test_percentage) / validation_percentage), random_state=RANDOM_SEED_VALUE, shuffle=True)
train_id, val_id = next(sgkf.split(train_val_df, train_val_df["Target"], groups=train_val_df["nih_patient_id"]))
train_df = train_val_df.iloc[train_id].copy()
val_df = train_val_df.iloc[val_id].copy()

print(f"Count size of splits.\n train: {len(train_df)}\n validation: {len(val_df)}\n test: {len(test_df)} \n total: {len(val_df) + len(train_df) + len(test_df)}")
print(f"Percentage size of splits.\n train: {len(train_df) / len(labels_df)}\n validation: {len(val_df) / len(labels_df)}\n test: {len(test_df) / len(labels_df)}")
print("#" * 40)

print(f"Class distributions per split.\n train:{train_df["Target"].value_counts(normalize=True)}\n validation:{val_df["Target"].value_counts(normalize=True)}\n test:{test_df["Target"].value_counts(normalize=True)}")

set_train_df = set(train_df["nih_patient_id"])
set_val_df = set(val_df["nih_patient_id"])
set_test_df = set(test_df["nih_patient_id"])


Count size of splits.
 train: 20327
 validation: 2664
 test: 2693 
 total: 25684
Percentage size of splits.
 train: 0.7914265690702382
 validation: 0.10372216165706276
 test: 0.10485126927269896
########################################
Class distributions per split.
 train:Target
0    0.777242
1    0.222758
Name: proportion, dtype: float64
 validation:Target
0    0.792042
1    0.207958
Name: proportion, dtype: float64
 test:Target
0    0.785741
1    0.214259
Name: proportion, dtype: float64


In [31]:
dicom_files = list(Path("dataset").rglob("*.dcm"))
print(pydicom.dcmread(dicom_files[0]))


mapping = {file.stem: file for file in dicom_files}


for d in (train_df, val_df, test_df):
    d["image_path"] = d["patientId"].map(mapping)

assert train_df["image_path"].notna().all()
assert test_df["image_path"].notna().all()
assert val_df["image_path"].notna().all()
assert set_train_df.isdisjoint(set_val_df)
assert set_train_df.isdisjoint(set_test_df)
assert set_val_df.isdisjoint(set_test_df)


#Now ALL i need is just the nih_pateint_id, Target and image location
train_df = train_df[["nih_patient_id", "Target", "image_path"]]
test_df = test_df[["nih_patient_id", "Target", "image_path"]]
val_df = val_df[["nih_patient_id", "Target", "image_path"]]

print(train_df.head())

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 202
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: Secondary Capture Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.2.276.0.7230010.3.1.4.8323329.13028.1517874368.999134
(0002,0010) Transfer Syntax UID                 UI: JPEG Baseline (Process 1)
(0002,0012) Implementation Class UID            UI: 1.2.276.0.7230010.3.0.3.6.0
(0002,0013) Implementation Version Name         SH: 'OFFIS_DCMTK_360'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0016) SOP Class UID                       UI: Secondary Capture Image Storage
(0008,0018) SOP Instance UID                    UI: 1.2.276.0.7230010.3.1.4.8323329.13028.1517874368.999134
(0008,0020) Study Date                          DA: '19010101'
(0008,0030) Study Time                  

## C

In [40]:
# My plan is to 1) Make a function that, given a path, retreives the image

def load_image(path):
    image = pydicom.dcmread(path, stop_before_pixels=False)
    image_pixels = image.pixel_array

    #Convert to torch tensor

    image_pixels = torch.from_numpy(image_pixels)
    image_pixels = torch.unsqueeze(image_pixels, 0)

    #Normalise values 
    amax, amin = image_pixels.max(), image_pixels.min()
    image_pixels = (image_pixels - amin) / (amax - amin)

    #Resize to 224 x 224

    resize_transform = v2.Resize((224, 224), antialias=True)
    image_pixels = resize_transform(image_pixels)

    return image_pixels


imagenet_mean=sum([0.485, 0.456, 0.406]) / 3
imagenet_std=sum([0.229, 0.224, 0.225]) / 3

#RStandardisation transform
normalize = v2.Normalize(mean=[imagenet_mean], std=[imagenet_std])

train_transform = v2.Compose([
    v2.RandomAffine(degrees=10, translate=(0.03, 0.03), interpolation=InterpolationMode.BILINEAR),
    v2.ColorJitter(brightness=0.1, contrast=0.1),
    normalize
])
val_test_transform = v2.Compose([normalize])



Make the datasets

In [46]:
class Pneumonia_Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        #For a given index, return the image and label in that order
        row = self.df.iloc[index]
        path = row["image_path"]
        image_pixels = load_image(path=path)
        label = row["Target"]
        if self.transform is not None:
            image_pixels = self.transform(image_pixels)

        #squeeze the extra dimension again
        image_pixels = image_pixels.squeeze(0)
        image_pixels = np.stack([image_pixels] * 3, axis=0)

        
        return torch.tensor(image_pixels, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)


#THE model datasets and dataloaders
train_ds = Pneumonia_Dataset(train_df, transform=train_transform)
val_ds = Pneumonia_Dataset(val_df, transform=val_test_transform)
test_ds = Pneumonia_Dataset(test_df, transform=val_test_transform)


train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)



        

## D

In [42]:
def trivial_baseline(df):
    mode_class = 0
    pred = len(df) * [mode_class]
    return classification_report(df["Target"], pred, zero_division=0)

print(f"Validation: {trivial_baseline(val_df)}")
print(f"Test: {trivial_baseline(test_df)}")


Validation:               precision    recall  f1-score   support

           0       0.79      1.00      0.88      2110
           1       0.00      0.00      0.00       554

    accuracy                           0.79      2664
   macro avg       0.40      0.50      0.44      2664
weighted avg       0.63      0.79      0.70      2664

Test:               precision    recall  f1-score   support

           0       0.79      1.00      0.88      2116
           1       0.00      0.00      0.00       577

    accuracy                           0.79      2693
   macro avg       0.39      0.50      0.44      2693
weighted avg       0.62      0.79      0.69      2693



Linear baseline

In [22]:
class Baseline_Dataset(Dataset):
    def __init__(self, df, ds_mean=None, ds_std=None):
        self.df = df.reset_index(drop=True)
        self.mean = ds_mean
        self.std = ds_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        path = row["image_path"]
        image_pixels = load_image(path=path)
        label = row["Target"]

        histogram, _ = np.histogram(image_pixels, bins=5, density=True)

        features = np.concatenate([[image_pixels.mean(), image_pixels.std()], histogram, np.percentile(image_pixels, [20, 40, 60, 80])], dtype=np.float32)

        if self.mean is not None and self.std is not None:
            features = (features - self.mean) / self.std



        
        return torch.tensor(features), torch.tensor(label, dtype=torch.float32)


#now get the mean and standard deviation for each
x = Baseline_Dataset(train_df)
nums = torch.stack([x[i][0] for i in range(len(x))])
mean = nums.mean(dim=0).numpy()
std = nums.std(dim=0).numpy() + 1e-8

print(f"mean: {mean}, std: {std}")

baseline_train_dataset = Baseline_Dataset(train_df, ds_mean=mean, ds_std=std)
baseline_val_dataset = Baseline_Dataset(val_df, ds_mean=mean, ds_std=std)
baseline_test_dataset = Baseline_Dataset(test_df, ds_mean=mean, ds_std=std)

def materialize(ds):
    features, labels = [], []
    for i in range(len(ds)):
        feature, label = ds[i]
        features.append(feature)
        labels.append(label)
    return torch.stack(features), torch.stack(labels)

X_train, y_train = materialize(baseline_train_dataset)
X_val, y_val     = materialize(baseline_val_dataset)
X_test, y_test   = materialize(baseline_test_dataset)

baseline_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
baseline_val_loader   = DataLoader(TensorDataset(X_val, y_val),     batch_size=32, shuffle=False)
baseline_test_loader  = DataLoader(TensorDataset(X_test, y_test),   batch_size=32, shuffle=False)



mean: [0.5056559  0.23393825 0.7431167  0.83682734 1.303248   1.4506134
 1.059147   0.29272917 0.4582477  0.5936848  0.72843987], std: [0.09179639 0.04045991 0.45752698 0.39018896 0.44324148 0.534393
 0.64431256 0.13045704 0.12051141 0.10646852 0.09193703]


In [23]:
class LogisticBaseline(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1) #We would get [batchsize, 1] by the end so just remove this 1


model = LogisticBaseline(num_features=11)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_func = nn.BCEWithLogitsLoss()

best_val_accuracy = 0
best_state = None

for epoch in range(50):
    model.train()
    for x, y in baseline_train_loader:
        optimizer.zero_grad()
        loss = loss_func(model(x), y)
        loss.backward()
        optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in baseline_val_loader:
            preds = (torch.sigmoid(model(x)) > 0.5).float()
            correct += (preds == y).sum().item()
            total += y.size(0)
    val_accuracy = correct / total

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_state = model.state_dict()

    print(f"Epoch {epoch+1}: val_accuracy={val_accuracy:.4f}")

model.load_state_dict(best_state)

Epoch 1: val_accuracy=0.7928
Epoch 2: val_accuracy=0.7875
Epoch 3: val_accuracy=0.7917
Epoch 4: val_accuracy=0.7935
Epoch 5: val_accuracy=0.7917
Epoch 6: val_accuracy=0.7898
Epoch 7: val_accuracy=0.7853
Epoch 8: val_accuracy=0.7872
Epoch 9: val_accuracy=0.7883
Epoch 10: val_accuracy=0.7909
Epoch 11: val_accuracy=0.7890
Epoch 12: val_accuracy=0.7898
Epoch 13: val_accuracy=0.7872
Epoch 14: val_accuracy=0.7902
Epoch 15: val_accuracy=0.7924
Epoch 16: val_accuracy=0.7879
Epoch 17: val_accuracy=0.7879
Epoch 18: val_accuracy=0.7887
Epoch 19: val_accuracy=0.7872
Epoch 20: val_accuracy=0.7827
Epoch 21: val_accuracy=0.7905
Epoch 22: val_accuracy=0.7864
Epoch 23: val_accuracy=0.7902
Epoch 24: val_accuracy=0.7812
Epoch 25: val_accuracy=0.7883
Epoch 26: val_accuracy=0.7857
Epoch 27: val_accuracy=0.7875
Epoch 28: val_accuracy=0.7838
Epoch 29: val_accuracy=0.7864
Epoch 30: val_accuracy=0.7883
Epoch 31: val_accuracy=0.7898
Epoch 32: val_accuracy=0.7920
Epoch 33: val_accuracy=0.7909
Epoch 34: val_accur

<All keys matched successfully>

In [24]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in baseline_test_loader:
        preds = (torch.sigmoid(model(x)) > 0.5).float()
        all_preds.append(preds)
        all_labels.append(y)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

print(classification_report(all_labels, all_preds, zero_division=0))

              precision    recall  f1-score   support

         0.0       0.80      0.97      0.88      2116
         1.0       0.49      0.09      0.16       577

    accuracy                           0.78      2693
   macro avg       0.64      0.53      0.52      2693
weighted avg       0.73      0.78      0.72      2693



## E

In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


In [ ]:
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 1) 
    return model

model = build_model().to(device)
model.train()
#STAGE 1 OF JUST TRAINING THE CLASSIFIER
for params in model.parameters():
    params.requires_grad = False

for params in model.fc.parameters():
    params.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
loss_func = nn.BCEWithLogitsLoss()


train_losses, val_losses = [], []
best_val_accuracy = 0
best_state = None

for epoch in range(10):
    model.train()
    running_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_func(model(x).squeeze(1), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
    train_losses.append(running_loss / len(train_loader.dataset))

    model.eval()
    val_running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            output = model(x).squeeze(1)
            val_running_loss += loss_func(output, y).item() * x.size(0)
            preds = (torch.sigmoid(output) > 0.5).float()
            correct += (preds == y).sum().item()
            total += y.size(0)
    val_losses.append(val_running_loss / len(val_loader.dataset))
    val_accuracy = correct / total

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_state = model.state_dict()

    print(f"Epoch {epoch+1}: val_accuracy={val_accuracy:.4f}")

model.load_state_dict(best_state)







Epoch 1: val_accuracy=0.8097
Epoch 2: val_accuracy=0.8179
Epoch 3: val_accuracy=0.8176
Epoch 4: val_accuracy=0.8086
Epoch 5: val_accuracy=0.8123
Epoch 6: val_accuracy=0.8191
Epoch 7: val_accuracy=0.8224
